# HAI Subjectivity Study — Updated Analysis (Directional Reliance + Revision Magnitude)

**Primary (confirmatory) family:** Directional reliance **R** on **Low vs High** (AI present), across 4 modalities, Holm-corrected.  
**Secondary family:** Revision magnitude **FF** on **Baseline vs Low vs High**, across 4 modalities (Holm omnibus + Holm post-hoc within modality).  
Includes effect sizes, cluster-bootstrap CIs, and non-parametric sensitivity checks.

Inputs expected in `BASE_DIR`:
- `comprehensive_theme_file2.csv`
- `comprehensive_language_file2.csv`
- `comprehensive_detail_lengths2.csv`
- `comprehensive_stance_file.csv`


In [1]:
# Cell 1 — Imports & paths
import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from joblib import Parallel, delayed
from pathlib import Path

BASE_DIR = Path.cwd()

THEME_PATH  = BASE_DIR / "data" / "comprehensive_theme_file2.csv"
LIWC_PATH   = BASE_DIR / "data" / "comprehensive_language_file2.csv"
DETAIL_PATH = BASE_DIR / "data" / "comprehensive_detail_lengths2.csv"
STANCE_PATH = BASE_DIR / "data" / "comprehensive_stance_file.csv"

OUT_DIR = BASE_DIR / "outputs" / "directionalR-revisionFF"

# Qualtrics raw exports and combined analysis file (gitignored — participant data)
PERCEIVED_PATH = BASE_DIR / "data" / "unified_theme_analysis_results_3.xlsx"
QUALTRICS_HIGH = BASE_DIR / "data" / "H-AI_Subjectivity_Study_High_Interaction_filtered.csv"
QUALTRICS_LOW  = BASE_DIR / "data" / "H-AI_Subjectivity_Study_Low_Interaction_filtered.csv"
QUALTRICS_BASE = BASE_DIR / "data" / "H-AI_Subjectivity_Study_Baseline_filtered.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [THEME_PATH, LIWC_PATH, DETAIL_PATH, STANCE_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing input: {p}")

print("OK — inputs found.")
print("Outputs →", OUT_DIR)


OK — inputs found.
Outputs → /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF


## Helpers (MixedLM, bootstrap CIs, effect sizes)


In [2]:
# Cell 2 — Helpers

def set_condition_order(df, order=("low","high","baseline")):
    df = df.copy()
    df["condition"] = df["condition"].astype(str).str.strip().str.lower()
    df["condition"] = pd.Categorical(df["condition"], categories=list(order), ordered=True)
    return df

def fit_mixedlm(df, formula, group_col="response_id", scenario_col="scenario"):
    d = df.copy()
    d[group_col] = d[group_col].astype(str)
    d[scenario_col] = d[scenario_col].astype(str)
    d["condition"] = d["condition"].astype("category")
    vc = {"scenario": f"0 + C({scenario_col})"}
    model = smf.mixedlm(formula, d, groups=d[group_col], vc_formula=vc, re_formula="1")
    res = model.fit(reml=False, method="lbfgs", maxiter=200, disp=False)
    return res

def cluster_bootstrap_ci(df, stat_func, cluster_col="response_id", n_boot=100, seed=123, n_jobs=-1):
    rng = np.random.default_rng(seed)
    clusters = df[cluster_col].dropna().unique()
    if len(clusters) < 5:
        return (np.nan, np.nan)
    # Pre-build lookup once; each draw renames its cluster to avoid duplicate IDs
    # in the mixed model (duplicate group IDs cause non-convergence and slow fits).
    cg = {cid: df[df[cluster_col] == cid] for cid in clusters}
    seeds = rng.integers(0, 2**31, size=n_boot)
    def _one(s):
        r = np.random.default_rng(s)
        samp = r.choice(clusters, size=len(clusters), replace=True)
        parts = []
        for new_id, orig_id in enumerate(samp):
            chunk = cg[orig_id].copy()
            chunk[cluster_col] = f"_b{new_id}"
            parts.append(chunk)
        boot = pd.concat(parts, ignore_index=True)
        return stat_func(boot)
    vals = Parallel(n_jobs=n_jobs)(delayed(_one)(s) for s in seeds)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)

def cohen_d_from_participant_means(df, outcome, group_col="condition", id_col="response_id",
                                  group_a="low", group_b="high"):
    dd = df[[id_col, group_col, outcome]].dropna().copy()
    dd[group_col] = dd[group_col].astype(str).str.lower()
    pm = dd.groupby([id_col, group_col])[outcome].mean().reset_index()
    a = pm.loc[pm[group_col]==group_a, outcome].to_numpy()
    b = pm.loc[pm[group_col]==group_b, outcome].to_numpy()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    sa = a.std(ddof=1); sb = b.std(ddof=1)
    sp = np.sqrt(((len(a)-1)*sa*sa + (len(b)-1)*sb*sb) / (len(a)+len(b)-2))
    if sp == 0:
        return np.nan
    return float((b.mean() - a.mean()) / sp)

def wald_omnibus_condition(res, terms):
    idx = list(res.params.index)
    present = [t for t in terms if t in idx]
    if not present:
        return np.nan
    R = np.zeros((len(present), len(idx)))
    for i,t in enumerate(present):
        R[i, idx.index(t)] = 1.0
    return float(res.wald_test(R).pvalue)


## Load and compute outcomes (R and FF)


In [3]:
# Cell 3 — Load + compute outcomes

theme = pd.read_csv(THEME_PATH, low_memory=False)
liwc  = pd.read_csv(LIWC_PATH, low_memory=False)
detail= pd.read_csv(DETAIL_PATH, low_memory=False)
stance= pd.read_csv(STANCE_PATH, low_memory=False)

def norm_keys(df):
    df = df.copy()
    for c in ["condition","response_id","scenario"]:
        if c not in df.columns:
            raise ValueError(f"Missing required key column '{c}'")
    df["condition"] = df["condition"].astype(str).str.strip().str.lower()
    df["response_id"] = df["response_id"].astype(str).str.strip()
    df["scenario"] = df["scenario"].astype(str).str.strip().str.lower()
    return df

theme = norm_keys(theme); liwc = norm_keys(liwc); detail = norm_keys(detail); stance = norm_keys(stance)

# Theme
req = {"cosine_first_ai","cosine_final_ai","cosine_first_final"}
if not req.issubset(theme.columns):
    raise ValueError("Theme file missing required cosine columns.")
theme["R"] = theme["cosine_final_ai"] - theme["cosine_first_ai"]
theme["FF"] = theme["cosine_first_final"]
theme_mod = theme[["response_id","scenario","condition","R","FF"]].copy()
theme_mod["modality"] = "theme"

# LIWC
req = {"cosine_first_ai_liwc","cosine_final_ai_liwc","cosine_first_final_liwc"}
if not req.issubset(liwc.columns):
    raise ValueError("LIWC file missing required cosine columns.")
liwc["R"] = liwc["cosine_final_ai_liwc"] - liwc["cosine_first_ai_liwc"]
liwc["FF"] = liwc["cosine_first_final_liwc"]
liwc_mod = liwc[["response_id","scenario","condition","R","FF"]].copy()
liwc_mod["modality"] = "liwc"

# Detail (words distance)
need_detail = {"dist_words_first_ai","dist_words_final_ai","delta_words_first_final"}
if not need_detail.issubset(detail.columns):
    raise ValueError(f"Detail missing: {sorted(need_detail - set(detail.columns))}")
detail["R"] = detail["dist_words_first_ai"] - detail["dist_words_final_ai"]
detail["FF"] = detail["delta_words_first_final"].abs()  # revision magnitude: absolute change in words
detail_mod = detail[["response_id","scenario","condition","R","FF"]].copy()
detail_mod["modality"] = "detail_words"

# Stance
ai_col = None
for c in ["ai_stance_final_norm","ai_stance","ai_stance_norm","ai_stance_norm_final"]:
    if c in stance.columns:
        ai_col = c; break
if ai_col is None:
    raise ValueError("Stance file missing AI stance column (ai_stance_final_norm / ai_stance / ai_stance_norm).")

def norm_stance(x):
    if pd.isna(x):
        return np.nan
    t = str(x).strip().lower()
    if "depend" in t or "unsure" in t or "maybe" in t:
        return "depends"
    if t.startswith("y") or t == "yes":
        return "yes"
    if t.startswith("n") or t == "no":
        return "no"
    return "depends"

first_col = "stance_first" if "stance_first" in stance.columns else ("first_stance" if "first_stance" in stance.columns else None)
final_col = "stance_final" if "stance_final" in stance.columns else ("final_stance" if "final_stance" in stance.columns else None)
if first_col is None or final_col is None:
    raise ValueError("Stance file missing stance_first/final columns.")

stance["first_norm"] = stance[first_col].apply(norm_stance)
stance["final_norm"] = stance[final_col].apply(norm_stance)
stance["ai_norm"] = stance[ai_col].apply(norm_stance)

map_num = {"no":0.0, "depends":0.5, "yes":1.0}
stance["first_num"] = stance["first_norm"].map(map_num)
stance["final_num"] = stance["final_norm"].map(map_num)
stance["ai_num"] = stance["ai_norm"].map(map_num)

stance["R"] = (stance["first_num"] - stance["ai_num"]).abs() - (stance["final_num"] - stance["ai_num"]).abs()
stance["FF"] = (stance["final_num"] - stance["first_num"]).abs()

stance_mod = stance[["response_id","scenario","condition","R","FF"]].copy()
stance_mod["modality"] = "stance"

all_mod = pd.concat([theme_mod, liwc_mod, detail_mod, stance_mod], ignore_index=True)
print("Combined rows:", len(all_mod))
display(all_mod.head())

all_mod.to_csv(OUT_DIR / "all_mod.csv", index=False)
print("Saved:", OUT_DIR / "all_mod.csv")


Combined rows: 4953


,response_id,scenario,condition,R,FF,modality
0,R_10SgfcVir4vmnBf,aita-1,baseline,NaN,NaN,theme
1,R_10SgfcVir4vmnBf,aita-2,baseline,NaN,0.435758,theme
2,R_10SgfcVir4vmnBf,aita-3,baseline,NaN,NaN,theme
3,R_10SgfcVir4vmnBf,aita-4,baseline,NaN,0.574663,theme
4,R_10SgfcVir4vmnBf,sexism-1,baseline,NaN,0.598560,theme


Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/all_mod.csv


In [4]:
# Cell 3b — Non-engager filter (high condition only)
# Excludes participants with < 2 turns AND < 20 total words (approved threshold).
# Mapping: data-derived/prolific_to_response_mapping.csv  (Qualtrics ResponseId ← Q84 column)
# Exclusion list: data-derived/exclude_words20.csv         (28 prolific_id + scenario_id pairs)

_MAPPING_PATH = BASE_DIR / "data-derived" / "prolific_to_response_mapping.csv"
_EXCL_PATH    = BASE_DIR / "data-derived" / "exclude_words20.csv"

_mapping = pd.read_csv(_MAPPING_PATH)
_mapping["prolific_id"] = _mapping["prolific_id"].astype(str).str.strip()
_mapping["response_id"] = _mapping["response_id"].astype(str).str.strip()
_mapping = _mapping.dropna(subset=["prolific_id", "response_id"])

_excl = pd.read_csv(_EXCL_PATH)
_excl["prolific_id"] = _excl["prolific_id"].astype(str).str.strip()
_excl["scenario_id"] = _excl["scenario_id"].astype(str).str.strip()

# Translate (prolific_id, scenario_id) → (response_id, scenario) and build exclusion set
_excl_rid = _excl.merge(_mapping[["prolific_id", "response_id"]], on="prolific_id", how="inner")
_excl_set = set(zip(_excl_rid["response_id"], _excl_rid["scenario_id"]))

_n_before = len(all_mod)
_mask = all_mod.apply(lambda r: (r["response_id"], r["scenario"]) in _excl_set, axis=1)
all_mod = all_mod[~_mask].copy()

_n_mapped   = len(_excl_rid)
_n_unmapped = len(_excl) - len(_excl.merge(_mapping[["prolific_id"]], on="prolific_id", how="inner").drop_duplicates("prolific_id"))
print(f"Engagement filter applied to high condition:")
print(f"  Exclusion pairs in list : {len(_excl)}")
print(f"  Mapped to response_id   : {_n_mapped}  (removed from all_mod)")
print(f"  No survey response      : {_n_unmapped}  (already absent from CSVs)")
print(f"  Rows: {_n_before} → {len(all_mod)}")
all_mod.to_csv(OUT_DIR / "all_mod_filtered.csv", index=False)


Engagement filter applied to high condition:
  Exclusion pairs in list : 28
  Mapped to response_id   : 21  (removed from all_mod)
  No survey response      : 17  (already absent from CSVs)
  Rows: 4953 → 4870


## Primary family: Directional reliance R (Low vs High) with Holm across modalities


In [5]:
# Cell 4 — Primary family: R on Low vs High

primary = all_mod[all_mod["condition"].isin(["low","high"])].copy()
primary = set_condition_order(primary, order=("low","high"))

mods = ["theme","liwc","stance"]
rows = []
for mod in mods:
    d = primary[primary["modality"]==mod].dropna(subset=["R"]).copy()
    if len(d)==0: 
        continue
    res = fit_mixedlm(d, "R ~ C(condition)")
    term = "C(condition)[T.high]"
    est = float(res.params.get(term, np.nan))
    p = float(res.pvalues.get(term, np.nan))
    def boot_stat(boot):
        bres = fit_mixedlm(boot, "R ~ C(condition)")
        return float(bres.params.get(term, np.nan))
    ci_lo, ci_hi = cluster_bootstrap_ci(d, boot_stat, n_boot=100, seed=100+hash(mod)%1000)
    d_eff = cohen_d_from_participant_means(d, "R", group_a="low", group_b="high")
    means = d.groupby("condition")["R"].mean().to_dict()
    rows.append({"modality":mod,"n_rows":len(d),"n_participants":d["response_id"].nunique(),
                 "estimate_high_minus_low":est,"ci_low":ci_lo,"ci_high":ci_hi,"p_raw":p,
                 "cohen_d_participant_means":d_eff,"means_by_condition":means})

primary_df = pd.DataFrame(rows)
rej, padj, _, _ = multipletests(primary_df["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
primary_df["p_holm_family"] = padj
primary_df["reject_holm_0.05"] = rej

primary_df.to_csv(OUT_DIR / "primary_directionalR_results.csv", index=False)
print("Saved:", OUT_DIR / "primary_directionalR_results.csv")
display(primary_df)


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warning

Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/primary_directionalR_results.csv


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,modality,n_rows,n_participants,estimate_high_minus_low,ci_low,ci_high,p_raw,cohen_d_participant_means,means_by_condition,p_holm_family,reject_holm_0.05
0,theme,459,119,-0.001353,-0.028803,0.028899,0.932470,-0.005723,"{'low': 0.013216481937340771, 'high': 0.011928...",1.00000,False
1,liwc,459,119,-0.002374,-0.018365,0.014647,0.760377,-0.046167,"{'low': 0.007057363267517854, 'high': 0.004652...",1.00000,False
2,stance,447,119,-0.019711,-0.055369,0.015022,0.292940,-0.215669,"{'low': 0.028761061946902654, 'high': 0.009049...",0.87882,False


PRIMARY FAMILY other version of reliance where it is FinalAI - firstfinal

In [6]:
# Cell X — Alternative reliance: FinalAI - FirstFinal (Low vs High)

primary_alt = all_mod[all_mod["condition"].isin(["low","high"])].copy()
primary_alt = set_condition_order(primary_alt, order=("low","high"))

# Build R_alt per modality from the source files (not from all_mod["R"]/["FF"])
# We'll reload the original modality files to access the needed columns.

theme_df = pd.read_csv(THEME_PATH, low_memory=False)
liwc_df  = pd.read_csv(LIWC_PATH, low_memory=False)
detail_df= pd.read_csv(DETAIL_PATH, low_memory=False)
stance_df= pd.read_csv(STANCE_PATH, low_memory=False)

def norm_keys_local(df):
    df = df.copy()
    df["condition"] = df["condition"].astype(str).str.strip().str.lower()
    df["response_id"] = df["response_id"].astype(str).str.strip()
    df["scenario"] = df["scenario"].astype(str).str.strip().str.lower()
    return df

theme_df = norm_keys_local(theme_df)
liwc_df  = norm_keys_local(liwc_df)
detail_df= norm_keys_local(detail_df)
stance_df= norm_keys_local(stance_df)

# Theme: FA - FF
theme_df["R_alt"] = theme_df["cosine_final_ai"] - theme_df["cosine_first_final"]
theme_alt = theme_df[["response_id","scenario","condition","R_alt"]].copy()
theme_alt["modality"] = "theme"

# LIWC: FA - FF
liwc_df["R_alt"] = liwc_df["cosine_final_ai_liwc"] - liwc_df["cosine_first_final_liwc"]
liwc_alt = liwc_df[["response_id","scenario","condition","R_alt"]].copy()
liwc_alt["modality"] = "liwc"

# Detail: use distances so units match
# D_FA = |final_words - ai_words|
# D_FF = |final_words - initial_words|
# R_alt = D_FF - D_FA  (positive => closer to AI than to initial, in length)

need_cols = {"initial_words", "final_words", "ai_words"}
if not need_cols.issubset(detail_df.columns):
    raise ValueError(f"Detail file missing required columns for absolute distances: {need_cols - set(detail_df.columns)}")

detail_df["D_FA"] = (detail_df["final_words"] - detail_df["ai_words"]).abs()
detail_df["D_FF"] = (detail_df["final_words"] - detail_df["initial_words"]).abs()
detail_df["R_alt"] = detail_df["D_FF"] - detail_df["D_FA"]

detail_alt = detail_df[["response_id","scenario","condition","R_alt"]].copy()
detail_alt["modality"] = "detail_words"

# Stance: use distance form so units match
# dist_final_ai = |final - AI|, dist_first_final = |final - first|
ai_col = None
for c in ["ai_stance_final_norm","ai_stance","ai_stance_norm","ai_stance_norm_final"]:
    if c in stance_df.columns:
        ai_col = c; break
if ai_col is None:
    raise ValueError("Stance file missing AI stance column.")

first_col = "stance_first" if "stance_first" in stance_df.columns else "first_stance"
final_col = "stance_final" if "stance_final" in stance_df.columns else "final_stance"

def norm_stance(x):
    if pd.isna(x):
        return np.nan
    t = str(x).strip().lower()
    if "depend" in t or "unsure" in t or "maybe" in t:
        return "depends"
    if t.startswith("y") or t == "yes":
        return "yes"
    if t.startswith("n") or t == "no":
        return "no"
    return "depends"

map_num = {"no":0.0, "depends":0.5, "yes":1.0}

stance_df["first_num"] = stance_df[first_col].apply(norm_stance).map(map_num)
stance_df["final_num"] = stance_df[final_col].apply(norm_stance).map(map_num)
stance_df["ai_num"]    = stance_df[ai_col].apply(norm_stance).map(map_num)

stance_df["dist_final_ai"] = (stance_df["final_num"] - stance_df["ai_num"]).abs()
stance_df["dist_first_final"] = (stance_df["final_num"] - stance_df["first_num"]).abs()

# Alternative: FF distance - FA distance (positive => closer to AI than to initial, in distance terms)
stance_df["R_alt"] = stance_df["dist_first_final"] - stance_df["dist_final_ai"]

stance_alt = stance_df[["response_id","scenario","condition","R_alt"]].copy()
stance_alt["modality"] = "stance"

# Combine
alt_all = pd.concat([theme_alt, liwc_alt, detail_alt, stance_alt], ignore_index=True)
alt_all = alt_all[alt_all["condition"].isin(["low","high"])].copy()
alt_all = set_condition_order(alt_all, order=("low","high"))

# Save row-level alternative reliance values for reuse (e.g., perceived reliance models)
ALT_ROWLEVEL_PATH = OUT_DIR / "rowlevel_alt_reliance_finalAI_minus_firstFinal.csv"
alt_all.to_csv(ALT_ROWLEVEL_PATH, index=False)
print("Saved row-level alt reliance:", ALT_ROWLEVEL_PATH)

mods = ["theme","liwc","stance"]
rows = []

for mod in mods:
    d = alt_all[alt_all["modality"]==mod].dropna(subset=["R_alt"]).copy()
    if len(d)==0:
        continue

    res = fit_mixedlm(d, "R_alt ~ C(condition)")
    term = "C(condition)[T.high]"
    est = float(res.params.get(term, np.nan))
    p = float(res.pvalues.get(term, np.nan))

    def boot_stat(boot):
        bres = fit_mixedlm(boot, "R_alt ~ C(condition)")
        return float(bres.params.get(term, np.nan))

    ci_lo, ci_hi = cluster_bootstrap_ci(d, boot_stat, n_boot=100, seed=200+hash(mod)%1000)

    d_eff = cohen_d_from_participant_means(d, "R_alt", group_a="low", group_b="high")

    means = d.groupby("condition")["R_alt"].mean().to_dict()
    rows.append({
        "modality": mod,
        "n_rows": len(d),
        "n_participants": d["response_id"].nunique(),
        "estimate_high_minus_low": est,
        "ci_low": ci_lo,
        "ci_high": ci_hi,
        "p_raw": p,
        "cohen_d_participant_means": d_eff,
        "means_by_condition": means
    })

alt_df = pd.DataFrame(rows)

# Holm across modalities (this is a sensitivity family)
rej, padj, _, _ = multipletests(alt_df["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
alt_df["p_holm_family"] = padj
alt_df["reject_holm_0.05"] = rej

alt_df.to_csv(OUT_DIR / "sensitivity_alt_reliance_finalAI_minus_firstFinal.csv", index=False)
print("Saved:", OUT_DIR / "sensitivity_alt_reliance_finalAI_minus_firstFinal.csv")
display(alt_df)

Saved row-level alt reliance: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/rowlevel_alt_reliance_finalAI_minus_firstFinal.csv


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarni

Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/sensitivity_alt_reliance_finalAI_minus_firstFinal.csv


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarni

,modality,n_rows,n_participants,estimate_high_minus_low,ci_low,ci_high,p_raw,cohen_d_participant_means,means_by_condition,p_holm_family,reject_holm_0.05
0,theme,479,120,-0.015393,-0.063410,0.025246,4.939346e-01,-0.123526,"{'low': -0.06728008194028466, 'high': -0.08271...",9.878692e-01,False
1,liwc,479,120,-0.075986,-0.094573,-0.054499,8.980163e-13,-1.297004,"{'low': -0.041562797839411386, 'high': -0.1175...",2.694049e-12,True
2,stance,467,120,0.021215,-0.030307,0.077444,5.196267e-01,0.159294,"{'low': -0.3075221238938053, 'high': -0.286307...",9.878692e-01,False


## Secondary family: Revision magnitude FF (Baseline vs Low vs High) omnibus + Holm across modalities


In [7]:
# Cell 5 — Secondary family: FF omnibus

secondary = all_mod[all_mod["condition"].isin(["baseline","low","high"])].copy()
secondary = set_condition_order(secondary, order=("low","high","baseline"))  # Low reference

mods = ["theme","liwc","detail_words","stance"]
rows=[]
for mod in mods:
    d = secondary[secondary["modality"]==mod].dropna(subset=["FF"]).copy()
    if len(d)==0:
        continue
    res = fit_mixedlm(d, "FF ~ C(condition)")
    p_omni = wald_omnibus_condition(res, ["C(condition)[T.high]","C(condition)[T.baseline]"])
    means = d.groupby("condition")["FF"].mean().to_dict()
    rows.append({"modality":mod,"n_rows":len(d),"n_participants":d["response_id"].nunique(),
                 "p_raw_omnibus":p_omni,"means_by_condition":means,
                 "coef_high_minus_low":float(res.params.get("C(condition)[T.high]", np.nan)),
                 "coef_baseline_minus_low":float(res.params.get("C(condition)[T.baseline]", np.nan))})

secondary_df = pd.DataFrame(rows)
rej, padj, _, _ = multipletests(secondary_df["p_raw_omnibus"].fillna(1.0).values, method="holm", alpha=0.05)
secondary_df["p_holm_family"] = padj
secondary_df["reject_holm_0.05"] = rej

secondary_df.to_csv(OUT_DIR / "secondary_revisionFF_omnibus.csv", index=False)
print("Saved:", OUT_DIR / "secondary_revisionFF_omnibus.csv")
display(secondary_df)


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWar

Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/secondary_revisionFF_omnibus.csv


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(


,modality,n_rows,n_participants,p_raw_omnibus,means_by_condition,coef_high_minus_low,coef_baseline_minus_low,p_holm_family,reject_holm_0.05
0,theme,687,176,0.004025,"{'low': 0.6884644671825536, 'high': 0.63433207...",-0.049351,0.042408,0.012074,True
1,liwc,687,176,0.000958,"{'low': 0.9187203488130645, 'high': 0.90382011...",-0.014191,0.015750,0.003830,True
2,detail_words,687,176,0.014398,"{'low': 11.093617021276597, 'high': 16.0401785...",4.864031,-0.135151,0.028795,True
3,stance,668,176,0.867331,"{'low': 0.046460176991150445, 'high': 0.054298...",0.007899,-0.001213,0.867331,False


## Secondary post-hoc: pairwise FF contrasts with Holm within modality + bootstrap CIs


In [8]:
# Cell 6 — Post-hoc FF pairwise within modality

def posthoc_pairwise_ff(d_mod):
    d_mod = set_condition_order(d_mod, order=("low","high","baseline"))
    res = fit_mixedlm(d_mod, "FF ~ C(condition)")

    b_hl = float(res.params.get("C(condition)[T.high]", np.nan))
    p_hl = float(res.pvalues.get("C(condition)[T.high]", np.nan))
    b_bl = float(res.params.get("C(condition)[T.baseline]", np.nan))
    p_bl = float(res.pvalues.get("C(condition)[T.baseline]", np.nan))

    # --- High vs Baseline (robust): refit with baseline as reference ---
    d_hb = d_mod.copy()
    d_hb["condition"] = pd.Categorical(d_hb["condition"], categories=["baseline","low","high"], ordered=True)
    res_hb = fit_mixedlm(d_hb, "FF ~ C(condition)")
    b_hb = float(res_hb.params.get("C(condition)[T.high]", np.nan))
    p_hb = float(res_hb.pvalues.get("C(condition)[T.high]", np.nan))

    def boot_high(boot):
        bres = fit_mixedlm(boot, "FF ~ C(condition)")
        return float(bres.params.get("C(condition)[T.high]", np.nan))

    def boot_base(boot):
        bres = fit_mixedlm(boot, "FF ~ C(condition)")
        return float(bres.params.get("C(condition)[T.baseline]", np.nan))

    def boot_hb(boot):
        boot2 = boot.copy()
        boot2["condition"] = pd.Categorical(boot2["condition"], categories=["baseline","low","high"], ordered=True)
        bres2 = fit_mixedlm(boot2, "FF ~ C(condition)")
        return float(bres2.params.get("C(condition)[T.high]", np.nan))

    lo_hl, hi_hl = cluster_bootstrap_ci(d_mod, boot_high, n_boot=100, seed=2024)
    lo_bl, hi_bl = cluster_bootstrap_ci(d_mod, boot_base, n_boot=100, seed=2025)
    lo_hb, hi_hb = cluster_bootstrap_ci(d_mod, boot_hb, n_boot=100, seed=2026)

    out = pd.DataFrame([
        {"contrast":"High vs Low","estimate":b_hl,"p_raw":p_hl,"ci_low":lo_hl,"ci_high":hi_hl},
        {"contrast":"Baseline vs Low","estimate":b_bl,"p_raw":p_bl,"ci_low":lo_bl,"ci_high":hi_bl},
        {"contrast":"High vs Baseline","estimate":b_hb,"p_raw":p_hb,"ci_low":lo_hb,"ci_high":hi_hb},
    ])
    rej, padj, _, _ = multipletests(out["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
    out["p_holm_within_modality"] = padj
    out["reject_holm_0.05"] = rej
    return out

posthoc=[]
for mod in ["theme","liwc","detail_words","stance"]:
    d = secondary[secondary["modality"]==mod].dropna(subset=["FF"]).copy()
    if len(d)==0: 
        continue
    tbl = posthoc_pairwise_ff(d)
    tbl.insert(0,"modality",mod)
    posthoc.append(tbl)

posthoc_df = pd.concat(posthoc, ignore_index=True)
posthoc_df.to_csv(OUT_DIR / "secondary_revisionFF_posthoc_pairwise.csv", index=False)
print("Saved:", OUT_DIR / "secondary_revisionFF_posthoc_pairwise.csv")
display(posthoc_df)


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarni

Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/secondary_revisionFF_posthoc_pairwise.csv


,modality,contrast,estimate,p_raw,ci_low,ci_high,p_holm_within_modality,reject_holm_0.05
0,theme,High vs Low,-0.049351,0.072030,-0.104726,0.004390,0.144060,False
1,theme,Baseline vs Low,0.042408,0.123900,-0.009245,0.099333,0.144060,False
2,theme,High vs Baseline,-0.091759,0.000909,-0.140998,-0.038403,0.002728,True
3,liwc,High vs Low,-0.014191,0.074991,-0.029569,0.004686,0.096525,False
4,liwc,Baseline vs Low,0.015750,0.048263,0.002959,0.032744,0.096525,False
5,liwc,High vs Baseline,-0.029941,0.000194,-0.044205,-0.015248,0.000581,True
6,detail_words,High vs Low,4.864031,0.012420,1.133285,9.629302,0.032402,True
7,detail_words,Baseline vs Low,-0.135151,0.944658,-2.527761,3.081819,0.944658,False
8,detail_words,High vs Baseline,4.999182,0.010801,0.694500,9.829165,0.032402,True
9,stance,High vs Low,0.007899,0.668855,-0.021730,0.052589,1.000000,False


SECONDARY x2 just checking finalAI

In [9]:
# Cell 5 (FinalAI) — Alignment omnibus (Low vs High only)

# Build a "FA" column for each modality from your source files already loaded earlier
# Assumes: theme, liwc, detail, stance were loaded in Cell 3 and keys normalized.

# Theme FA: cosine_final_ai
theme_FA = theme[["response_id","scenario","condition","cosine_final_ai"]].rename(columns={"cosine_final_ai":"FA"}).copy()
theme_FA["modality"] = "theme"

# LIWC FA: cosine_final_ai_liwc
liwc_FA = liwc[["response_id","scenario","condition","cosine_final_ai_liwc"]].rename(columns={"cosine_final_ai_liwc":"FA"}).copy()
liwc_FA["modality"] = "liwc"

# Detail FA: distance final to AI in words (already in file)
detail_FA = detail[["response_id","scenario","condition","dist_words_final_ai"]].rename(columns={"dist_words_final_ai":"FA"}).copy()
detail_FA["modality"] = "detail_words"

# Stance FA: distance final to AI on 0/0.5/1 scale
stance_FA = stance[["response_id","scenario","condition"]].copy()
stance_FA["FA"] = (stance["final_num"] - stance["ai_num"]).abs()
stance_FA["modality"] = "stance"

fa_all = pd.concat([theme_FA, liwc_FA, detail_FA, stance_FA], ignore_index=True)

# Only Low/High (baseline has no AI)
secondary_fa = fa_all[fa_all["condition"].isin(["low","high"])].copy()
secondary_fa = set_condition_order(secondary_fa, order=("low","high"))  # Low reference

mods = ["theme","liwc","stance"]
rows = []
for mod in mods:
    d = secondary_fa[secondary_fa["modality"]==mod].dropna(subset=["FA"]).copy()
    if len(d)==0:
        continue

    res = fit_mixedlm(d, "FA ~ C(condition)")
    term = "C(condition)[T.high]"
    est = float(res.params.get(term, np.nan))
    p = float(res.pvalues.get(term, np.nan))  # with 2 levels, this is the omnibus too
    means = d.groupby("condition")["FA"].mean().to_dict()

    rows.append({
        "modality": mod,
        "n_rows": len(d),
        "n_participants": d["response_id"].nunique(),
        "p_raw_omnibus": p,
        "means_by_condition": means,
        "coef_high_minus_low": est
    })

secondary_fa_df = pd.DataFrame(rows)

# Holm across modalities (this is a sensitivity family unless you declare it confirmatory)
rej, padj, _, _ = multipletests(secondary_fa_df["p_raw_omnibus"].fillna(1.0).values, method="holm", alpha=0.05)
secondary_fa_df["p_holm_family"] = padj
secondary_fa_df["reject_holm_0.05"] = rej

secondary_fa_df.to_csv(OUT_DIR / "sensitivity_finalAI_alignment_omnibus.csv", index=False)
print("Saved:", OUT_DIR / "sensitivity_finalAI_alignment_omnibus.csv")
display(secondary_fa_df)

/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/sensitivity_finalAI_alignment_omnibus.csv


,modality,n_rows,n_participants,p_raw_omnibus,means_by_condition,coef_high_minus_low,p_holm_family,reject_holm_0.05
0,theme,480,120,3.238994e-03,"{'low': 0.6204776527809825, 'high': 0.55866623...",-0.061811,6.477988e-03,True
1,liwc,480,120,7.382852e-16,"{'low': 0.8773423300676793, 'high': 0.78842249...",-0.088920,2.214856e-15,True
2,stance,468,120,6.790852e-01,"{'low': 0.35398230088495575, 'high': 0.3429752...",-0.011007,6.790852e-01,False


In [10]:
# Cell 6 (FinalAI) — Alignment pairwise (High vs Low only) with bootstrap CI
# (refit with explicit categorical levels to avoid missing-term / boundary issues)

def pairwise_fa(d_mod):
    d_mod = d_mod.copy()
    d_mod["condition"] = d_mod["condition"].astype(str).str.strip().str.lower()
    d_mod["condition"] = pd.Categorical(d_mod["condition"], categories=["low","high"], ordered=True)

    res = fit_mixedlm(d_mod, "FA ~ C(condition)")
    term = "C(condition)[T.high]"

    b_hl = float(res.params.get(term, np.nan))
    p_hl = float(res.pvalues.get(term, np.nan))

    def boot_hl(boot):
        boot2 = boot.copy()
        boot2["condition"] = boot2["condition"].astype(str).str.strip().str.lower()
        boot2["condition"] = pd.Categorical(boot2["condition"], categories=["low","high"], ordered=True)
        bres = fit_mixedlm(boot2, "FA ~ C(condition)")
        return float(bres.params.get(term, np.nan))

    lo_hl, hi_hl = cluster_bootstrap_ci(d_mod, boot_hl, n_boot=100, seed=3030)

    out = pd.DataFrame([{
        "contrast": "High vs Low",
        "estimate": b_hl,
        "p_raw": p_hl,
        "ci_low": lo_hl,
        "ci_high": hi_hl
    }])
    return out

posthoc = []
for mod in ["theme","liwc","stance"]:
    d = secondary_fa[secondary_fa["modality"]==mod].dropna(subset=["FA"]).copy()
    if len(d)==0:
        continue
    tbl = pairwise_fa(d)
    tbl.insert(0, "modality", mod)
    posthoc.append(tbl)

fa_pairwise_df = pd.concat(posthoc, ignore_index=True)

# Holm across modalities (family-level correction)
rej, padj, _, _ = multipletests(fa_pairwise_df["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
fa_pairwise_df["p_holm_across_modalities"] = padj
fa_pairwise_df["reject_holm_0.05"] = rej

fa_pairwise_df.to_csv(OUT_DIR / "sensitivity_finalAI_alignment_pairwise.csv", index=False)
print("Saved:", OUT_DIR / "sensitivity_finalAI_alignment_pairwise.csv")
display(fa_pairwise_df)

/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarni

Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/sensitivity_finalAI_alignment_pairwise.csv


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the pa

,modality,contrast,estimate,p_raw,ci_low,ci_high,p_holm_across_modalities,reject_holm_0.05
0,theme,High vs Low,-0.061811,3.238994e-03,-0.103863,-0.023431,6.477988e-03,True
1,liwc,High vs Low,-0.088920,7.382852e-16,-0.108587,-0.066571,2.214856e-15,True
2,stance,High vs Low,-0.011007,6.790852e-01,-0.053643,0.030175,6.790852e-01,False


## Robustness: participant aggregation + non-parametric sensitivity checks


In [11]:
# Cell 7 — Sensitivity checks

agg = (all_mod.groupby(["modality","condition","response_id"], as_index=False)
            .agg(R_mean=("R","mean"), FF_mean=("FF","mean")))

# Directional R: MWU low vs high
sens_R=[]
for mod in ["theme","liwc","stance"]:
    dd = agg[(agg["modality"]==mod) & (agg["condition"].isin(["low","high"]))].dropna(subset=["R_mean"])
    a = dd.loc[dd["condition"]=="low","R_mean"].to_numpy()
    b = dd.loc[dd["condition"]=="high","R_mean"].to_numpy()
    if len(a)<2 or len(b)<2:
        p=np.nan
    else:
        _, p = st.mannwhitneyu(a,b,alternative="two-sided")
    sens_R.append({"modality":mod,"p_raw":float(p) if p==p else np.nan,
                   "mean_low":float(np.nanmean(a)) if len(a) else np.nan,
                   "mean_high":float(np.nanmean(b)) if len(b) else np.nan})
sens_R = pd.DataFrame(sens_R)
rej, padj, _, _ = multipletests(sens_R["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
sens_R["p_holm_across_modalities"] = padj
sens_R["reject_holm_0.05"] = rej
sens_R.to_csv(OUT_DIR / "sensitivity_directionalR_MWU.csv", index=False)
display(sens_R)

# Revision FF: KW + pairwise MWU within modality
sens_FF=[]
pairwise=[]
for mod in ["theme","liwc","stance"]:
    dd = agg[(agg["modality"]==mod) & (agg["condition"].isin(["baseline","low","high"]))].dropna(subset=["FF_mean"])
    groups = [dd.loc[dd["condition"]==c,"FF_mean"].to_numpy() for c in ["baseline","low","high"]]
    if any(len(g)<2 for g in groups):
        p_kw=np.nan
    else:
        _, p_kw = st.kruskal(*groups)
    sens_FF.append({"modality":mod,"p_raw":float(p_kw) if p_kw==p_kw else np.nan})

    comps=[("baseline","low"),("baseline","high"),("low","high")]
    pvals=[]
    rows=[]
    for a,b in comps:
        x = dd.loc[dd["condition"]==a,"FF_mean"].to_numpy()
        y = dd.loc[dd["condition"]==b,"FF_mean"].to_numpy()
        if len(x)<2 or len(y)<2:
            p=np.nan
        else:
            _, p = st.mannwhitneyu(x,y,alternative="two-sided")
        pvals.append(1.0 if not (p==p) else float(p))
        rows.append({"modality":mod,"comparison":f"{a} vs {b}","p_raw":float(p) if p==p else np.nan})
    rej, padj, _, _ = multipletests(np.array(pvals), method="holm", alpha=0.05)
    for i,r in enumerate(rows):
        r["p_holm_within_modality"]=float(padj[i])
        r["reject_holm_0.05"]=bool(rej[i])
    pairwise.extend(rows)

sens_FF=pd.DataFrame(sens_FF)
rej, padj, _, _ = multipletests(sens_FF["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
sens_FF["p_holm_across_modalities"]=padj
sens_FF["reject_holm_0.05"]=rej
sens_FF.to_csv(OUT_DIR / "sensitivity_revisionFF_KW.csv", index=False)
pd.DataFrame(pairwise).to_csv(OUT_DIR / "sensitivity_revisionFF_pairwise_MWU.csv", index=False)

display(sens_FF)
display(pd.DataFrame(pairwise).head(12))


,modality,p_raw,mean_low,mean_high,p_holm_across_modalities,reject_holm_0.05
0,theme,0.557001,0.012914,0.012406,1.0,False
1,liwc,0.696057,0.007095,0.005116,1.0,False
2,stance,0.402599,0.027542,0.006250,1.0,False


,modality,p_raw,p_holm_across_modalities,reject_holm_0.05
0,theme,0.008637,0.017273,True
1,liwc,0.001814,0.005441,True
2,stance,0.772306,0.772306,False


,modality,comparison,p_raw,p_holm_within_modality,reject_holm_0.05
0,theme,baseline vs low,0.141838,0.159798,False
1,theme,baseline vs high,0.002732,0.008195,True
2,theme,low vs high,0.079899,0.159798,False
3,liwc,baseline vs low,0.098693,0.116243,False
4,liwc,baseline vs high,0.000397,0.001192,True
5,liwc,low vs high,0.058122,0.116243,False
6,stance,baseline vs low,0.889863,1.000000,False
7,stance,baseline vs high,0.503955,1.000000,False
8,stance,low vs high,0.598345,1.000000,False


In [12]:
# Cell 7A — Sensitivity checks for Relative Alignment (RA): MWU low vs high

import numpy as np
import pandas as pd
import scipy.stats as st
from statsmodels.stats.multitest import multipletests

# alt_all is the long row-level table with columns: response_id, scenario, condition, modality, R_alt
assert "alt_all" in globals(), "alt_all not found. Run the alt reliance cell first (creates alt_all)."

alt = alt_all.copy()
alt["condition"] = alt["condition"].astype(str).str.strip().str.lower()
alt["response_id"] = alt["response_id"].astype(str).str.strip()
alt["scenario"] = alt["scenario"].astype(str).str.strip().str.lower()

# Aggregate per participant within modality and condition (mean across scenarios)
agg_ra = (alt[alt["modality"].isin(["theme","liwc","stance"]) & alt["condition"].isin(["low","high"])]
          .groupby(["modality","condition","response_id"], as_index=False)
          .agg(RA_mean=("R_alt","mean")))

sens_RA = []
for mod in ["theme","liwc","stance"]:
    dd = agg_ra[(agg_ra["modality"]==mod)].dropna(subset=["RA_mean"])
    a = dd.loc[dd["condition"]=="low", "RA_mean"].to_numpy()
    b = dd.loc[dd["condition"]=="high","RA_mean"].to_numpy()
    if len(a) < 2 or len(b) < 2:
        p = np.nan
    else:
        _, p = st.mannwhitneyu(a, b, alternative="two-sided")

    sens_RA.append({
        "modality": mod,
        "p_raw": float(p) if p==p else np.nan,
        "mean_low": float(np.nanmean(a)) if len(a) else np.nan,
        "mean_high": float(np.nanmean(b)) if len(b) else np.nan
    })

sens_RA = pd.DataFrame(sens_RA)

# Holm across modalities for this family (RA sensitivity)
rej, padj, _, _ = multipletests(sens_RA["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
sens_RA["p_holm_across_modalities"] = padj
sens_RA["reject_holm_0.05"] = rej

sens_RA.to_csv(OUT_DIR / "sensitivity_relativeAlignment_RA_MWU.csv", index=False)
print("Saved:", OUT_DIR / "sensitivity_relativeAlignment_RA_MWU.csv")
display(sens_RA)

Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/sensitivity_relativeAlignment_RA_MWU.csv


,modality,p_raw,mean_low,mean_high,p_holm_across_modalities,reject_holm_0.05
0,theme,5.356361e-01,-0.067380,-0.082712,5.525331e-01,False
1,liwc,2.393987e-09,-0.041461,-0.117527,7.181960e-09,True
2,stance,2.762665e-01,-0.312147,-0.287568,5.525331e-01,False


In [13]:
# Cell 7B — Sensitivity checks for FinalAI alignment (FA): MWU low vs high

import numpy as np
import pandas as pd
import scipy.stats as st
from statsmodels.stats.multitest import multipletests

# secondary_fa is the long table with columns: response_id, scenario, condition, modality, FA
assert "secondary_fa" in globals(), "secondary_fa not found. Run the FinalAI construction cell first (creates secondary_fa)."

fa = secondary_fa.copy()
fa["condition"] = fa["condition"].astype(str).str.strip().str.lower()
fa["response_id"] = fa["response_id"].astype(str).str.strip()
fa["scenario"] = fa["scenario"].astype(str).str.strip().str.lower()

# Aggregate per participant within modality and condition (mean across scenarios)
agg_fa = (fa[fa["modality"].isin(["theme","liwc","stance"]) & fa["condition"].isin(["low","high"])]
          .groupby(["modality","condition","response_id"], as_index=False)
          .agg(FA_mean=("FA","mean")))

sens_FA = []
for mod in ["theme","liwc","stance"]:
    dd = agg_fa[(agg_fa["modality"]==mod)].dropna(subset=["FA_mean"])
    a = dd.loc[dd["condition"]=="low", "FA_mean"].to_numpy()
    b = dd.loc[dd["condition"]=="high","FA_mean"].to_numpy()
    if len(a) < 2 or len(b) < 2:
        p = np.nan
    else:
        _, p = st.mannwhitneyu(a, b, alternative="two-sided")

    sens_FA.append({
        "modality": mod,
        "p_raw": float(p) if p==p else np.nan,
        "mean_low": float(np.nanmean(a)) if len(a) else np.nan,
        "mean_high": float(np.nanmean(b)) if len(b) else np.nan
    })

sens_FA = pd.DataFrame(sens_FA)

# Holm across modalities for this family (FA sensitivity)
rej, padj, _, _ = multipletests(sens_FA["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
sens_FA["p_holm_across_modalities"] = padj
sens_FA["reject_holm_0.05"] = rej

sens_FA.to_csv(OUT_DIR / "sensitivity_finalAI_FA_MWU.csv", index=False)
print("Saved:", OUT_DIR / "sensitivity_finalAI_FA_MWU.csv")
display(sens_FA)

Saved: /Users/jeevanparmar/Uni/Research/Ferguson/Human-AI-Reliance-Paper-Code/Post-Study-Analysis/outputs_updated_directionalR_revisionFF/sensitivity_finalAI_FA_MWU.csv


,modality,p_raw,mean_low,mean_high,p_holm_across_modalities,reject_holm_0.05
0,theme,8.027028e-03,0.620478,0.558666,1.605406e-02,True
1,liwc,3.240605e-11,0.877342,0.788422,9.721814e-11,True
2,stance,6.080737e-01,0.356638,0.344945,6.080737e-01,False


In [14]:
import numpy as np
import pandas as pd
import scipy.stats as st
from statsmodels.stats.multitest import multipletests

# --- Make sure secondary_fa has clean columns (not index levels) ---
sec = secondary_fa.copy()

# If any of these are index levels, bring them back as columns
sec = sec.reset_index(drop=False)

# If reset_index created duplicates like 'level_0', drop them
for junk in ["index", "level_0"]:
    if junk in sec.columns and junk not in ["response_id","scenario","condition","modality"]:
        sec = sec.drop(columns=[junk])

# Force required columns to exist and be 1D
required = ["modality","condition","response_id","FA"]
missing = [c for c in required if c not in sec.columns]
if missing:
    raise ValueError(f"secondary_fa is missing columns: {missing}. Columns are: {sec.columns.tolist()}")

# Clean types
sec["modality"] = sec["modality"].astype(str)
sec["condition"] = sec["condition"].astype(str).str.strip().str.lower()
sec["response_id"] = sec["response_id"].astype(str).str.strip()

# Now aggregate
agg_fa = (sec.groupby(["modality","condition","response_id"], as_index=False)["FA"]
            .mean()
            .rename(columns={"FA":"FA_mean"}))

# MWU low vs high per modality
sens_FA = []
for mod in ["theme","liwc","stance"]:
    dd = agg_fa[(agg_fa["modality"]==mod) & (agg_fa["condition"].isin(["low","high"]))].dropna(subset=["FA_mean"])
    a = dd.loc[dd["condition"]=="low", "FA_mean"].to_numpy()
    b = dd.loc[dd["condition"]=="high","FA_mean"].to_numpy()

    if len(a) < 2 or len(b) < 2:
        p = np.nan
    else:
        _, p = st.mannwhitneyu(a, b, alternative="two-sided")

    sens_FA.append({
        "modality": mod,
        "test": "MWU low vs high",
        "p_raw": float(p) if p==p else np.nan,
        "mean_low": float(np.nanmean(a)) if len(a) else np.nan,
        "mean_high": float(np.nanmean(b)) if len(b) else np.nan
    })

sens_FA = pd.DataFrame(sens_FA)

# Holm across modalities
rej, padj, _, _ = multipletests(sens_FA["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
sens_FA["p_holm_across_modalities"] = padj
sens_FA["reject_holm_0.05"] = rej

sens_FA.to_csv(OUT_DIR / "sensitivity_finalAI_MWU.csv", index=False)
display(sens_FA)

,modality,test,p_raw,mean_low,mean_high,p_holm_across_modalities,reject_holm_0.05
0,theme,MWU low vs high,8.027028e-03,0.620478,0.558666,1.605406e-02,True
1,liwc,MWU low vs high,3.240605e-11,0.877342,0.788422,9.721814e-11,True
2,stance,MWU low vs high,6.080737e-01,0.356638,0.344945,6.080737e-01,False


In [15]:
# Cell — Master sensitivity checks (R, RA, FA, FF) with clear labels

import numpy as np
import pandas as pd
import scipy.stats as st
from statsmodels.stats.multitest import multipletests

# ------------------------
# Helpers
# ------------------------
def _norm_df(df):
    df = df.copy()
    df["condition"] = df["condition"].astype(str).str.strip().str.lower()
    df["response_id"] = df["response_id"].astype(str).str.strip()
    if "scenario" in df.columns:
        df["scenario"] = df["scenario"].astype(str).str.strip().str.lower()
    df["modality"] = df["modality"].astype(str).str.strip().str.lower()
    return df

def mwu_low_high(dd, value_col):
    """Return (p, mean_low, mean_high) for MWU low vs high on dd[value_col]."""
    a = dd.loc[dd["condition"]=="low", value_col].to_numpy()
    b = dd.loc[dd["condition"]=="high", value_col].to_numpy()
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nanmean(a) if len(a) else np.nan, np.nanmean(b) if len(b) else np.nan
    _, p = st.mannwhitneyu(a, b, alternative="two-sided")
    return float(p), float(np.nanmean(a)), float(np.nanmean(b))

def holm_across_modalities(df, p_col="p_raw"):
    rej, padj, _, _ = multipletests(df[p_col].fillna(1.0).values, method="holm", alpha=0.05)
    df = df.copy()
    df["p_holm_across_modalities"] = padj
    df["reject_holm_0.05"] = rej
    return df

# ------------------------
# Check prerequisites
# ------------------------
assert "all_mod" in globals(), "Missing all_mod."
assert "alt_all" in globals(), "Missing alt_all (relative alignment row-level)."
assert "secondary_fa" in globals(), "Missing secondary_fa (final-AI alignment row-level)."

# Normalize
allm = _norm_df(all_mod)
alt  = _norm_df(alt_all)
fa   = _norm_df(secondary_fa)

# Restrict to modalities you analyze (exclude detail_words for AI-involving families if desired)
MODS_CORE = ["theme","liwc","stance"]

# ------------------------
# 1) Sensitivity: Directional reliance (movement) R = FinalAI - FirstAI
# Participant-aggregated MWU low vs high, Holm across modalities
# ------------------------
agg_R = (allm[allm["modality"].isin(MODS_CORE) & allm["condition"].isin(["low","high"])]
         .groupby(["modality","condition","response_id"], as_index=False)
         .agg(value=("R","mean"))
        )

rows=[]
for mod in MODS_CORE:
    dd = agg_R[(agg_R["modality"]==mod)].dropna(subset=["value"])
    p, mlow, mhigh = mwu_low_high(dd, "value")
    rows.append({"family":"Directional reliance (R = FinalAI - FirstAI)",
                 "outcome":"R_mean_participant",
                 "modality":mod,
                 "test":"MWU low vs high",
                 "p_raw":p,
                 "mean_low":mlow,
                 "mean_high":mhigh,
                 "n_participants_low":dd.loc[dd["condition"]=="low","response_id"].nunique(),
                 "n_participants_high":dd.loc[dd["condition"]=="high","response_id"].nunique()
                })
sens_R = holm_across_modalities(pd.DataFrame(rows))

print("\n=== Sensitivity A: Directional reliance (movement) R = FinalAI - FirstAI (MWU + Holm across modalities) ===")
display(sens_R)

sens_R.to_csv(OUT_DIR / "SENS_A_directionalR_MWU.csv", index=False)

# ------------------------
# 2) Sensitivity: Relative alignment RA = FinalAI - FirstFinal  (your R_alt)
# Participant-aggregated MWU low vs high, Holm across modalities
# ------------------------
agg_RA = (alt[alt["modality"].isin(MODS_CORE) & alt["condition"].isin(["low","high"])]
          .groupby(["modality","condition","response_id"], as_index=False)
          .agg(value=("R_alt","mean"))
         )

rows=[]
for mod in MODS_CORE:
    dd = agg_RA[(agg_RA["modality"]==mod)].dropna(subset=["value"])
    p, mlow, mhigh = mwu_low_high(dd, "value")
    rows.append({"family":"Relative alignment (RA = FinalAI - FirstFinal)",
                 "outcome":"RA_mean_participant",
                 "modality":mod,
                 "test":"MWU low vs high",
                 "p_raw":p,
                 "mean_low":mlow,
                 "mean_high":mhigh,
                 "n_participants_low":dd.loc[dd["condition"]=="low","response_id"].nunique(),
                 "n_participants_high":dd.loc[dd["condition"]=="high","response_id"].nunique()
                })
sens_RA = holm_across_modalities(pd.DataFrame(rows))

print("\n=== Sensitivity B: Relative alignment RA = FinalAI - FirstFinal (MWU + Holm across modalities) ===")
display(sens_RA)

sens_RA.to_csv(OUT_DIR / "SENS_B_relativeAlignment_RA_MWU.csv", index=False)

# ------------------------
# 3) Sensitivity: FinalAI alignment FA (final vs AI)
# Participant-aggregated MWU low vs high, Holm across modalities
# ------------------------
agg_FA = (fa[fa["modality"].isin(MODS_CORE) & fa["condition"].isin(["low","high"])]
          .groupby(["modality","condition","response_id"], as_index=False)
          .agg(value=("FA","mean"))
         )

rows=[]
for mod in MODS_CORE:
    dd = agg_FA[(agg_FA["modality"]==mod)].dropna(subset=["value"])
    p, mlow, mhigh = mwu_low_high(dd, "value")
    rows.append({"family":"FinalAI alignment (FA = sim(final,AI) or stance distance)",
                 "outcome":"FA_mean_participant",
                 "modality":mod,
                 "test":"MWU low vs high",
                 "p_raw":p,
                 "mean_low":mlow,
                 "mean_high":mhigh,
                 "n_participants_low":dd.loc[dd["condition"]=="low","response_id"].nunique(),
                 "n_participants_high":dd.loc[dd["condition"]=="high","response_id"].nunique()
                })
sens_FA = holm_across_modalities(pd.DataFrame(rows))

print("\n=== Sensitivity C: FinalAI alignment FA (MWU + Holm across modalities) ===")
display(sens_FA)

sens_FA.to_csv(OUT_DIR / "SENS_C_finalAI_FA_MWU.csv", index=False)

# ------------------------
# 4) Sensitivity: FirstFinal revision magnitude FF
# Omnibus: Kruskal–Wallis across baseline/low/high per modality + Holm across modalities
# Post-hoc: pairwise MWU within modality + Holm within modality
# ------------------------
agg_FF = (allm[allm["modality"].isin(MODS_CORE + ["detail_words"]) & allm["condition"].isin(["baseline","low","high"])]
          .groupby(["modality","condition","response_id"], as_index=False)
          .agg(value=("FF","mean"))
         )

# Omnibus per modality
rows=[]
pairwise_rows=[]
mods_ff = ["theme","liwc","detail_words","stance"]  # include detail_words here if you want human-only revision sensitivity
for mod in mods_ff:
    dd = agg_FF[(agg_FF["modality"]==mod)].dropna(subset=["value"])
    groups = [dd.loc[dd["condition"]==c, "value"].to_numpy() for c in ["baseline","low","high"]]
    if any(len(g) < 2 for g in groups):
        p_kw = np.nan
    else:
        _, p_kw = st.kruskal(*groups)
    rows.append({"family":"Revision magnitude (FF = sim(first,final) or word delta)",
                 "outcome":"FF_mean_participant",
                 "modality":mod,
                 "test":"Kruskal–Wallis baseline vs low vs high",
                 "p_raw":float(p_kw) if p_kw==p_kw else np.nan,
                 "n_participants":dd["response_id"].nunique()
                })

    # Pairwise MWU + Holm within modality
    comps = [("baseline","low"),("baseline","high"),("low","high")]
    pvals=[]
    tmp=[]
    for a,b in comps:
        x = dd.loc[dd["condition"]==a, "value"].to_numpy()
        y = dd.loc[dd["condition"]==b, "value"].to_numpy()
        if len(x) < 2 or len(y) < 2:
            p = np.nan
        else:
            _, p = st.mannwhitneyu(x, y, alternative="two-sided")
        pvals.append(1.0 if not (p==p) else float(p))
        tmp.append({"family":"Revision magnitude (FF)",
                    "modality":mod,
                    "comparison":f"{a} vs {b}",
                    "p_raw":float(p) if p==p else np.nan,
                    "mean_"+a: float(np.nanmean(x)) if len(x) else np.nan,
                    "mean_"+b: float(np.nanmean(y)) if len(y) else np.nan
                   })
    rej, padj, _, _ = multipletests(np.array(pvals), method="holm", alpha=0.05)
    for i,r in enumerate(tmp):
        r["p_holm_within_modality"] = float(padj[i])
        r["reject_holm_0.05"] = bool(rej[i])
    pairwise_rows.extend(tmp)

sens_FF_omnibus = holm_across_modalities(pd.DataFrame(rows))

print("\n=== Sensitivity D1: FirstFinal revision magnitude FF (Kruskal–Wallis omnibus + Holm across modalities) ===")
display(sens_FF_omnibus)
sens_FF_omnibus.to_csv(OUT_DIR / "SENS_D1_revisionFF_KW_omnibus.csv", index=False)

sens_FF_pairwise = pd.DataFrame(pairwise_rows)

print("\n=== Sensitivity D2: FirstFinal revision magnitude FF (pairwise MWU + Holm within modality) ===")
display(sens_FF_pairwise.head(20))
sens_FF_pairwise.to_csv(OUT_DIR / "SENS_D2_revisionFF_pairwise_MWU.csv", index=False)

print("\n✅ Master sensitivity checks saved to outputs folder.")



=== Sensitivity A: Directional reliance (movement) R = FinalAI - FirstAI (MWU + Holm across modalities) ===


,family,outcome,modality,test,p_raw,mean_low,mean_high,n_participants_low,n_participants_high,p_holm_across_modalities,reject_holm_0.05
0,Directional reliance (R = FinalAI - FirstAI),R_mean_participant,theme,MWU low vs high,0.557001,0.012914,0.012406,59,60,1.0,False
1,Directional reliance (R = FinalAI - FirstAI),R_mean_participant,liwc,MWU low vs high,0.696057,0.007095,0.005116,59,60,1.0,False
2,Directional reliance (R = FinalAI - FirstAI),R_mean_participant,stance,MWU low vs high,0.402599,0.027542,0.006250,59,60,1.0,False



=== Sensitivity B: Relative alignment RA = FinalAI - FirstFinal (MWU + Holm across modalities) ===


,family,outcome,modality,test,p_raw,mean_low,mean_high,n_participants_low,n_participants_high,p_holm_across_modalities,reject_holm_0.05
0,Relative alignment (RA = FinalAI - FirstFinal),RA_mean_participant,theme,MWU low vs high,5.356361e-01,-0.067380,-0.082712,59,61,5.525331e-01,False
1,Relative alignment (RA = FinalAI - FirstFinal),RA_mean_participant,liwc,MWU low vs high,2.393987e-09,-0.041461,-0.117527,59,61,7.181960e-09,True
2,Relative alignment (RA = FinalAI - FirstFinal),RA_mean_participant,stance,MWU low vs high,2.762665e-01,-0.312147,-0.287568,59,61,5.525331e-01,False



=== Sensitivity C: FinalAI alignment FA (MWU + Holm across modalities) ===


...


=== Sensitivity D1: FirstFinal revision magnitude FF (Kruskal–Wallis omnibus + Holm across modalities) ===


,family,outcome,modality,test,p_raw,n_participants,p_holm_across_modalities,reject_holm_0.05
0,"Revision magnitude (FF = sim(first,final) or w...",FF_mean_participant,theme,Kruskal–Wallis baseline vs low vs high,0.008637,176,0.025910,True
1,"Revision magnitude (FF = sim(first,final) or w...",FF_mean_participant,liwc,Kruskal–Wallis baseline vs low vs high,0.001814,176,0.007255,True
2,"Revision magnitude (FF = sim(first,final) or w...",FF_mean_participant,detail_words,Kruskal–Wallis baseline vs low vs high,0.090809,176,0.181618,False
3,"Revision magnitude (FF = sim(first,final) or w...",FF_mean_participant,stance,Kruskal–Wallis baseline vs low vs high,0.772306,176,0.772306,False



=== Sensitivity D2: FirstFinal revision magnitude FF (pairwise MWU + Holm within modality) ===


,family,modality,comparison,p_raw,mean_baseline,mean_low,p_holm_within_modality,reject_holm_0.05,mean_high
0,Revision magnitude (FF),theme,baseline vs low,0.141838,0.730727,0.688250,0.159798,False,NaN
1,Revision magnitude (FF),theme,baseline vs high,0.002732,0.730727,NaN,0.008195,True,0.642932
2,Revision magnitude (FF),theme,low vs high,0.079899,NaN,0.688250,0.159798,False,0.642932
3,Revision magnitude (FF),liwc,baseline vs low,0.098693,0.934468,0.918715,0.116243,False,NaN
4,Revision magnitude (FF),liwc,baseline vs high,0.000397,0.934468,NaN,0.001192,True,0.905543
5,Revision magnitude (FF),liwc,low vs high,0.058122,NaN,0.918715,0.116243,False,0.905543
6,Revision magnitude (FF),detail_words,baseline vs low,0.434401,10.947368,11.067797,0.434401,False,NaN
7,Revision magnitude (FF),detail_words,baseline vs high,0.037204,10.947368,NaN,0.111612,False,15.808333
8,Revision magnitude (FF),detail_words,low vs high,0.138024,NaN,11.067797,0.276048,False,15.808333
9,Revision magnitude (FF),stance,baseline vs low,0.889863,0.043860,0.044492,1.000000,False,NaN



✅ Master sensitivity checks saved to outputs folder.


PERCIEVED RELIANCE

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

PERCEIVED_XLSX = PERCEIVED_PATH  # set in Cell 1
perceived_raw = pd.read_excel(PERCEIVED_XLSX)

# Apply non-engager filter to perceived reliance data (high condition only)
_excl_map = pd.read_csv(BASE_DIR / "data-derived" / "prolific_to_response_mapping.csv")
_excl_map["prolific_id"] = _excl_map["prolific_id"].astype(str).str.strip()
_excl_map["response_id"] = _excl_map["response_id"].astype(str).str.strip()
_excl_list = pd.read_csv(BASE_DIR / "data-derived" / "exclude_words20.csv")
_excl_list["prolific_id"] = _excl_list["prolific_id"].astype(str).str.strip()
_excl_list["scenario_id"] = _excl_list["scenario_id"].astype(str).str.strip()
_perc_excl = _excl_list.merge(_excl_map[["prolific_id","response_id"]], on="prolific_id", how="inner")
_perc_excl_set = set(zip(_perc_excl["response_id"], _perc_excl["scenario_id"]))
_perc_before = len(perceived_raw)
perceived_raw = perceived_raw[~perceived_raw.apply(
    lambda r: (str(r.get("response_id","")).strip(), str(r.get("scenario","")).strip()) in _perc_excl_set, axis=1
)].copy()
print(f"Perceived reliance filter: {_perc_before} → {len(perceived_raw)} rows ({_perc_before-len(perceived_raw)} removed)")


# Identify the last two columns as perceived reliance measures
last2 = list(perceived_raw.columns[-2:])
print("Assuming perceived reliance columns are:", last2)

# Rename to stable names
perceived = perceived_raw.copy()
perceived = perceived.rename(columns={
    last2[0]: "perceived_0_100",
    last2[1]: "perceived_1_5"
})

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

rid_col = find_col(perceived, ["response_id","ResponseId","ResponseID","responseId"])
scen_col = find_col(perceived, ["scenario","scenario_id","Scenario","scenario_key","scenario_key_human"])
cond_col = find_col(perceived, ["condition","Condition"])

if rid_col is None or scen_col is None or cond_col is None:
    raise ValueError(f"Could not find join keys. Found rid={rid_col}, scen={scen_col}, cond={cond_col}. "
                     f"Available columns: {perceived.columns.tolist()}")

perceived = perceived.rename(columns={rid_col:"response_id", scen_col:"scenario", cond_col:"condition"}).copy()

# Drop empty scenario rows (no initial text)
text_col = None
for c in ["initial_text", "first_text", "first_explanation", "first_rationale", "text_first"]:
    if c in perceived.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError(f"Couldn't find an initial-text column to filter on. Available columns: {perceived.columns.tolist()}")

before = len(perceived)
perceived = perceived.dropna(subset=[text_col]).copy()
# also drop whitespace-only strings
perceived = perceived[perceived[text_col].astype(str).str.strip() != ""].copy()
after = len(perceived)

print(f"Dropped {before-after} rows with missing/empty {text_col}. Remaining: {after}")

# Normalize keys
perceived["response_id"] = perceived["response_id"].astype(str).str.strip()
perceived["scenario"] = perceived["scenario"].astype(str).str.strip().str.lower()
perceived["condition"] = perceived["condition"].astype(str).str.strip().str.lower()

# Keep only low/high (baseline has no perceived reliance)
perceived = perceived[perceived["condition"].isin(["low","high"])].copy()

# ---- Map perceived 1–5 words -> numeric ----
label_map = {
    "not at all": 1,
    "too little": 2,
    "the right amount": 3,
    "too much": 4,
    "completely": 5,
}

def map_perceived_1_5(x):
    if pd.isna(x):
        return np.nan
    # If it's already numeric (1-5), keep it
    try:
        v = float(x)
        if v.is_integer() and 1 <= int(v) <= 5:
            return int(v)
    except Exception:
        pass
    # Otherwise map text
    t = str(x).strip().lower()
    return label_map.get(t, np.nan)

perceived["perceived_1_5_num"] = perceived["perceived_1_5"].apply(map_perceived_1_5)

# Optional quick audit
print("Perceived 1–5 (raw) unique:", sorted(perceived["perceived_1_5"].dropna().astype(str).unique())[:20])
print("Perceived 1–5 (num) unique:", sorted(perceived["perceived_1_5_num"].dropna().unique()))

# Keep only what we need
perceived = perceived[["response_id","scenario","condition","perceived_0_100","perceived_1_5_num"]].copy()

print("Perceived rows:", len(perceived))
display(perceived.head())

# ---- Build observed R wide ----
use_modalities = ["theme","liwc","stance"]

obsR_wide = (
    all_mod[all_mod["modality"].isin(use_modalities)]
    .pivot_table(index=["response_id","scenario","condition"], columns="modality", values="R", aggfunc="first")
    .reset_index()
    .rename(columns={"theme":"R_theme", "liwc":"R_liwc", "stance":"R_stance"})
)

# Restrict to low/high for consistency
obsR_wide = obsR_wide[obsR_wide["condition"].isin(["low","high"])].copy()

print("Observed R wide rows:", len(obsR_wide))
display(obsR_wide.head())

# ---- Merge ----
merged = perceived.merge(obsR_wide, on=["response_id","scenario","condition"], how="left", validate="m:1")

print("Merged rows:", len(merged))
print("Missing R_theme:", merged["R_theme"].isna().mean())
print("Missing R_liwc:", merged["R_liwc"].isna().mean())
print("Missing R_stance:", merged["R_stance"].isna().mean())
display(merged.head())

Perceived reliance filter: 1536 → 1514 rows (22 removed)
Assuming perceived reliance columns are: ['reliance_scale', 'reliance_measure']
Dropped 767 rows with missing/empty initial_text. Remaining: 747
Perceived 1–5 (raw) unique: ['Completely', 'Not at all', 'The right amount', 'Too little', 'Too much']
Perceived 1–5 (num) unique: [1.0, 2.0, 3.0, 4.0, 5.0]
Perceived rows: 499


,response_id,scenario,condition,perceived_0_100,perceived_1_5_num
496,R_13ycIx8of2YAXcm,aita-1,high,NaN,NaN
499,R_13ycIx8of2YAXcm,aita-4,high,NaN,NaN
501,R_13ycIx8of2YAXcm,sexism-2,high,NaN,NaN
502,R_13ycIx8of2YAXcm,sexism-3,high,NaN,NaN
504,R_1EsOi0uitnZWBk4,aita-1,high,NaN,NaN


Observed R wide rows: 460


modality,response_id,scenario,condition,R_liwc,R_stance,R_theme
0,R_13ycIx8of2YAXcm,aita-1,high,0.010164,0.0,-0.013443
1,R_13ycIx8of2YAXcm,aita-4,high,0.058684,0.0,0.107646
2,R_13ycIx8of2YAXcm,sexism-2,high,0.080786,0.0,-0.220785
3,R_13ycIx8of2YAXcm,sexism-3,high,0.094641,0.0,0.060761
4,R_1EcJDMskY8YSDst,aita-1,low,-0.039789,0.0,-0.095530


Merged rows: 499
Missing R_theme: 0.0
Missing R_liwc: 0.0
Missing R_stance: 0.03006012024048096


,response_id,scenario,condition,perceived_0_100,perceived_1_5_num,R_liwc,R_stance,R_theme
0,R_13ycIx8of2YAXcm,aita-1,high,NaN,NaN,0.010164,0.0,-0.013443
1,R_13ycIx8of2YAXcm,aita-4,high,NaN,NaN,0.058684,0.0,0.107646
2,R_13ycIx8of2YAXcm,sexism-2,high,NaN,NaN,0.080786,0.0,-0.220785
3,R_13ycIx8of2YAXcm,sexism-3,high,NaN,NaN,0.094641,0.0,0.060761
4,R_1EsOi0uitnZWBk4,aita-1,high,NaN,NaN,-0.034748,0.0,0.186486


In [17]:
from statsmodels.stats.multitest import multipletests

# We'll use MixedLM:
# perceived ~ condition + (1|participant) + (1|scenario)

df_cond = merged.dropna(subset=["perceived_0_100","perceived_1_5_num"]).copy()
df_cond["condition"] = df_cond["condition"].astype(str).str.strip().str.lower()
df_cond = set_condition_order(df_cond, order=("low","high"))  # all 3 possible

# If baseline is present, great. If not, categories still work.
print(df_cond["condition"].value_counts(dropna=False))

cond_results = []

# Model A: 0-100 perceived reliance
res_pct = fit_mixedlm(df_cond.dropna(subset=["perceived_0_100"]),
                      "perceived_0_100 ~ C(condition)")
p_pct = wald_omnibus_condition(res_pct, ["C(condition)[T.low]", "C(condition)[T.high]"])
cond_results.append({"dv":"perceived_0_100", "model":"MixedLM", "p_raw_omnibus":p_pct})

# Model B: 1-7 perceived reliance (treat as approx continuous)
res_lik = fit_mixedlm(df_cond.dropna(subset=["perceived_1_5_num"]),
                      "perceived_1_5_num ~ C(condition)")
p_lik = wald_omnibus_condition(res_lik, ["C(condition)[T.low]", "C(condition)[T.high]"])
cond_results.append({"dv":"perceived_1_5_num", "model":"MixedLM", "p_raw_omnibus":p_lik})

cond_results = pd.DataFrame(cond_results)

# Holm across the 2 perceived DVs
rej, padj, _, _ = multipletests(cond_results["p_raw_omnibus"].fillna(1.0).values, method="holm", alpha=0.05)
cond_results["p_holm_family"] = padj
cond_results["reject_holm_0.05"] = rej

display(cond_results)

# Optional: report means by condition for interpretability
means_table = df_cond.groupby("condition")[["perceived_0_100","perceived_1_5_num"]].mean()
display(means_table)

cond_results.to_csv(OUT_DIR / "perceived_by_condition_models.csv", index=False)
means_table.to_csv(OUT_DIR / "perceived_by_condition_means.csv")
print("Saved perceived-by-condition outputs.")

condition
low     128
high    115
Name: count, dtype: int64


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1906: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavio

,dv,model,p_raw_omnibus,p_holm_family,reject_holm_0.05
0,perceived_0_100,MixedLM,0.465664,0.505131,False
1,perceived_1_5_num,MixedLM,0.252566,0.505131,False


,perceived_0_100,perceived_1_5_num
condition,,
low,32.187500,2.437500
high,34.521739,2.208696


Saved perceived-by-condition outputs.


In [18]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

df_pred = merged.copy()

# Keep rows with perceived + all predictors
df_pred = df_pred.dropna(subset=["perceived_0_100","perceived_1_5_num","R_theme","R_liwc","R_stance"]).copy()
df_pred["condition"] = df_pred["condition"].astype(str).str.strip().str.lower()
df_pred = set_condition_order(df_pred, order=("baseline","low","high"))

# Standardize predictors (z-score) for interpretable coefficient magnitudes
for c in ["R_stance","R_theme","R_liwc"]:
    mu = df_pred[c].mean()
    sd = df_pred[c].std(ddof=1)
    df_pred[c + "_z"] = (df_pred[c] - mu) / sd if sd != 0 else np.nan

# Model formula (add condition as a covariate; you can drop it if you want pure association)
formula_pct = "perceived_0_100 ~ R_stance_z + R_theme_z + R_liwc_z + C(condition)"
formula_lik = "perceived_1_5_num ~ R_stance_z + R_theme_z + R_liwc_z + C(condition)"

res_pct = fit_mixedlm(df_pred, formula_pct)
res_lik = fit_mixedlm(df_pred, formula_lik)

# Extract p-values for the 3 predictors from each model
pred_terms = ["R_stance_z", "R_theme_z", "R_liwc_z"]

rows = []
for dv, res in [("perceived_0_100", res_pct), ("perceived_1_5_num", res_lik)]:
    for t in pred_terms:
        rows.append({
            "dv": dv,
            "predictor": t.replace("_z",""),
            "estimate": float(res.params.get(t, np.nan)),
            "p_raw": float(res.pvalues.get(t, np.nan))
        })

pred_results = pd.DataFrame(rows)

# Holm across 6 tests (3 predictors × 2 DVs)
rej, padj, _, _ = multipletests(pred_results["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
pred_results["p_holm_family"] = padj
pred_results["reject_holm_0.05"] = rej

display(pred_results)

pred_results.to_csv(OUT_DIR / "perceived_by_observedR_models.csv", index=False)
print("Saved perceived-by-observedR outputs.")

/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,dv,predictor,estimate,p_raw,p_holm_family,reject_holm_0.05
0,perceived_0_100,R_stance,-0.688928,0.654859,1.0,False
1,perceived_0_100,R_theme,0.756940,0.660063,1.0,False
2,perceived_0_100,R_liwc,0.803570,0.652332,1.0,False
3,perceived_1_5_num,R_stance,-0.054782,0.371190,1.0,False
4,perceived_1_5_num,R_theme,0.066157,0.330359,1.0,False
5,perceived_1_5_num,R_liwc,0.078720,0.255157,1.0,False


Saved perceived-by-observedR outputs.


perceived reliance for R_alt

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

PERCEIVED_XLSX = PERCEIVED_PATH  # set in Cell 1
perceived_raw = pd.read_excel(PERCEIVED_XLSX)

# Identify the last two columns as perceived reliance measures
last2 = list(perceived_raw.columns[-2:])
print("Assuming perceived reliance columns are:", last2)

# Rename to stable names
perceived = perceived_raw.copy()
perceived = perceived.rename(columns={
    last2[0]: "perceived_0_100",
    last2[1]: "perceived_1_5"
})

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

rid_col = find_col(perceived, ["response_id","ResponseId","ResponseID","responseId"])
scen_col = find_col(perceived, ["scenario","scenario_id","Scenario","scenario_key","scenario_key_human"])
cond_col = find_col(perceived, ["condition","Condition"])

if rid_col is None or scen_col is None or cond_col is None:
    raise ValueError(f"Could not find join keys. Found rid={rid_col}, scen={scen_col}, cond={cond_col}. "
                     f"Available columns: {perceived.columns.tolist()}")

perceived = perceived.rename(columns={rid_col:"response_id", scen_col:"scenario", cond_col:"condition"}).copy()

# Drop empty scenario rows (no initial text)
text_col = None
for c in ["initial_text", "first_text", "first_explanation", "first_rationale", "text_first"]:
    if c in perceived.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError(f"Couldn't find an initial-text column to filter on. Available columns: {perceived.columns.tolist()}")

before = len(perceived)
perceived = perceived.dropna(subset=[text_col]).copy()
# also drop whitespace-only strings
perceived = perceived[perceived[text_col].astype(str).str.strip() != ""].copy()
after = len(perceived)

print(f"Dropped {before-after} rows with missing/empty {text_col}. Remaining: {after}")

# Normalize keys
perceived["response_id"] = perceived["response_id"].astype(str).str.strip()
perceived["scenario"] = perceived["scenario"].astype(str).str.strip().str.lower()
perceived["condition"] = perceived["condition"].astype(str).str.strip().str.lower()

# Keep only low/high (baseline has no perceived reliance)
perceived = perceived[perceived["condition"].isin(["low","high"])].copy()

# ---- Map perceived 1–5 words -> numeric ----
label_map = {
    "not at all": 1,
    "too little": 2,
    "the right amount": 3,
    "too much": 4,
    "completely": 5,
}

def map_perceived_1_5(x):
    if pd.isna(x):
        return np.nan
    # If it's already numeric (1-5), keep it
    try:
        v = float(x)
        if v.is_integer() and 1 <= int(v) <= 5:
            return int(v)
    except Exception:
        pass
    # Otherwise map text
    t = str(x).strip().lower()
    return label_map.get(t, np.nan)

perceived["perceived_1_5_num"] = perceived["perceived_1_5"].apply(map_perceived_1_5)

# Optional quick audit
print("Perceived 1–5 (raw) unique:", sorted(perceived["perceived_1_5"].dropna().astype(str).unique())[:20])
print("Perceived 1–5 (num) unique:", sorted(perceived["perceived_1_5_num"].dropna().unique()))

# Keep only what we need
perceived = perceived[["response_id","scenario","condition","perceived_0_100","perceived_1_5_num"]].copy()

print("Perceived rows:", len(perceived))
display(perceived.head())

# ---- Build observed R wide ----
use_modalities = ["theme","liwc","stance"]

obsR_wide = (
    all_mod[all_mod["modality"].isin(use_modalities)]
    .pivot_table(index=["response_id","scenario","condition"], columns="modality", values="R", aggfunc="first")
    .reset_index()
    .rename(columns={"theme":"R_theme", "liwc":"R_liwc", "stance":"R_stance"})
)

# Restrict to low/high for consistency
obsR_wide = obsR_wide[obsR_wide["condition"].isin(["low","high"])].copy()

print("Observed R wide rows:", len(obsR_wide))
display(obsR_wide.head())

# ---- Merge ----
merged = perceived.merge(obsR_wide, on=["response_id","scenario","condition"], how="left", validate="m:1")

print("Merged rows:", len(merged))
print("Missing R_theme:", merged["R_theme"].isna().mean())
print("Missing R_liwc:", merged["R_liwc"].isna().mean())
print("Missing R_stance:", merged["R_stance"].isna().mean())
display(merged.head())

Assuming perceived reliance columns are: ['reliance_scale', 'reliance_measure']
Dropped 769 rows with missing/empty initial_text. Remaining: 767
Perceived 1–5 (raw) unique: ['Completely', 'Not at all', 'The right amount', 'Too little', 'Too much']
Perceived 1–5 (num) unique: [1.0, 2.0, 3.0, 4.0, 5.0]
Perceived rows: 519


,response_id,scenario,condition,perceived_0_100,perceived_1_5_num
496,R_13ycIx8of2YAXcm,aita-1,high,NaN,NaN
499,R_13ycIx8of2YAXcm,aita-4,high,NaN,NaN
501,R_13ycIx8of2YAXcm,sexism-2,high,NaN,NaN
502,R_13ycIx8of2YAXcm,sexism-3,high,NaN,NaN
504,R_1EsOi0uitnZWBk4,aita-1,high,NaN,NaN


Observed R wide rows: 460


modality,response_id,scenario,condition,R_liwc,R_stance,R_theme
0,R_13ycIx8of2YAXcm,aita-1,high,0.010164,0.0,-0.013443
1,R_13ycIx8of2YAXcm,aita-4,high,0.058684,0.0,0.107646
2,R_13ycIx8of2YAXcm,sexism-2,high,0.080786,0.0,-0.220785
3,R_13ycIx8of2YAXcm,sexism-3,high,0.094641,0.0,0.060761
4,R_1EcJDMskY8YSDst,aita-1,low,-0.039789,0.0,-0.095530


Merged rows: 519
Missing R_theme: 0.038535645472061654
Missing R_liwc: 0.038535645472061654
Missing R_stance: 0.0674373795761079


,response_id,scenario,condition,perceived_0_100,perceived_1_5_num,R_liwc,R_stance,R_theme
0,R_13ycIx8of2YAXcm,aita-1,high,NaN,NaN,0.010164,0.0,-0.013443
1,R_13ycIx8of2YAXcm,aita-4,high,NaN,NaN,0.058684,0.0,0.107646
2,R_13ycIx8of2YAXcm,sexism-2,high,NaN,NaN,0.080786,0.0,-0.220785
3,R_13ycIx8of2YAXcm,sexism-3,high,NaN,NaN,0.094641,0.0,0.060761
4,R_1EsOi0uitnZWBk4,aita-1,high,NaN,NaN,-0.034748,0.0,0.186486


In [20]:
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

# perceived ~ condition + (1|participant) + (1|scenario)

df_cond = merged.dropna(subset=["perceived_0_100","perceived_1_5_num"]).copy()
df_cond["condition"] = df_cond["condition"].astype(str).str.strip().str.lower()

# Force exactly low/high categories so the coefficient is stable
df_cond["condition"] = pd.Categorical(df_cond["condition"], categories=["low","high"], ordered=True)

print(df_cond["condition"].value_counts(dropna=False))

cond_results = []

# Model A: 0-100 perceived reliance
d_pct = df_cond.dropna(subset=["perceived_0_100"]).copy()
res_pct = fit_mixedlm(d_pct, "perceived_0_100 ~ C(condition)")
term = "C(condition)[T.high]"  # High vs Low
p_pct = float(res_pct.pvalues.get(term, np.nan))
est_pct = float(res_pct.params.get(term, np.nan))
cond_results.append({"dv":"perceived_0_100", "model":"MixedLM",
                     "estimate_high_minus_low": est_pct, "p_raw": p_pct})

# Model B: 1-5 perceived reliance (treat as approx continuous)
d_lik = df_cond.dropna(subset=["perceived_1_5_num"]).copy()
res_lik = fit_mixedlm(d_lik, "perceived_1_5_num ~ C(condition)")
p_lik = float(res_lik.pvalues.get(term, np.nan))
est_lik = float(res_lik.params.get(term, np.nan))
cond_results.append({"dv":"perceived_1_5_num", "model":"MixedLM",
                     "estimate_high_minus_low": est_lik, "p_raw": p_lik})

cond_results = pd.DataFrame(cond_results)

# Holm across the 2 perceived DVs
rej, padj, _, _ = multipletests(cond_results["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
cond_results["p_holm_family"] = padj
cond_results["reject_holm_0.05"] = rej

display(cond_results)

# Means by condition for interpretability
means_table = df_cond.groupby("condition")[["perceived_0_100","perceived_1_5_num"]].mean()
display(means_table)

cond_results.to_csv(OUT_DIR / "perceived_by_condition_models.csv", index=False)
means_table.to_csv(OUT_DIR / "perceived_by_condition_means.csv")
print("Saved perceived-by-condition outputs.")

condition
low     128
high    127
Name: count, dtype: int64


/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1635: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,dv,model,estimate_high_minus_low,p_raw,p_holm_family,reject_holm_0.05
0,perceived_0_100,MixedLM,4.316158,0.431113,0.633726,False
1,perceived_1_5_num,MixedLM,-0.188621,0.316863,0.633726,False


,perceived_0_100,perceived_1_5_num
condition,,
low,32.187500,2.437500
high,34.566929,2.244094


Saved perceived-by-condition outputs.


In [21]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

df_pred = merged.copy()

# Keep rows with perceived + all predictors (low/high only)
df_pred["condition"] = df_pred["condition"].astype(str).str.strip().str.lower()
df_pred = df_pred[df_pred["condition"].isin(["low","high"])].copy()

df_pred = df_pred.dropna(subset=[
    "perceived_0_100", "perceived_1_5_num",
    "R_theme", "R_liwc", "R_stance"
]).copy()

# Force consistent condition categories (low reference)
df_pred["condition"] = pd.Categorical(df_pred["condition"], categories=["low","high"], ordered=True)

# Standardize predictors (z-score) so coefficients are comparable
for c in ["R_stance","R_theme","R_liwc"]:
    mu = df_pred[c].mean()
    sd = df_pred[c].std(ddof=1)
    df_pred[c + "_z"] = (df_pred[c] - mu) / sd if sd and sd != 0 else np.nan

# Models (include condition as a covariate; coefficient is High vs Low)
formula_pct = "perceived_0_100 ~ R_stance_z + R_theme_z + R_liwc_z + C(condition)"
formula_lik = "perceived_1_5_num ~ R_stance_z + R_theme_z + R_liwc_z + C(condition)"

res_pct = fit_mixedlm(df_pred, formula_pct)
res_lik = fit_mixedlm(df_pred, formula_lik)

pred_terms = ["R_stance_z", "R_theme_z", "R_liwc_z"]

rows = []
for dv, res in [("perceived_0_100", res_pct), ("perceived_1_5_num", res_lik)]:
    for t in pred_terms:
        rows.append({
            "dv": dv,
            "predictor": t.replace("_z",""),
            "estimate": float(res.params.get(t, np.nan)),
            "p_raw": float(res.pvalues.get(t, np.nan))
        })

pred_results = pd.DataFrame(rows)

# Holm across 6 tests (3 predictors × 2 DVs)
rej, padj, _, _ = multipletests(pred_results["p_raw"].fillna(1.0).values, method="holm", alpha=0.05)
pred_results["p_holm_family"] = padj
pred_results["reject_holm_0.05"] = rej

display(pred_results)

pred_results.to_csv(OUT_DIR / "perceived_by_observedR_models.csv", index=False)
print("Saved perceived-by-observedR outputs.")

/opt/homebrew/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2238: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,dv,predictor,estimate,p_raw,p_holm_family,reject_holm_0.05
0,perceived_0_100,R_stance,-0.688928,0.654859,1.0,False
1,perceived_0_100,R_theme,0.756940,0.660063,1.0,False
2,perceived_0_100,R_liwc,0.803570,0.652332,1.0,False
3,perceived_1_5_num,R_stance,-0.054782,0.371192,1.0,False
4,perceived_1_5_num,R_theme,0.066157,0.330357,1.0,False
5,perceived_1_5_num,R_liwc,0.078720,0.255158,1.0,False


Saved perceived-by-observedR outputs.
